# Syria Population Windowed Reading Map (2026)

Overview

Demonstrates windowed reading of the WorldPop 2026 population raster for Syria. A 3,000 × 3,000 pixel window is read from the upper-left section of the source raster without loading the complete dataset. The selected window primarily covers northwestern Syria.

WorldPop 2026の人口ラスタを対象に、必要な範囲だけを読み込むWindowed Readingを実行します。元ラスタ全体を読み込まず、左上から3,000 × 3,000ピクセルを取得し、主にシリア北西部の人口分布を表示します。

Objectives

- Read a selected raster window without loading the complete dataset
- Preserve NoData areas as transparent pixels
- Display estimated population per source grid cell
- Apply a custom colour map
- Display the selected window with administrative boundaries
- Export the completed map as an HTML file

- 人口ラスタ全体を読み込まず、指定した範囲だけを取得する
- NoData領域を透明な画素として保持する
- 元ラスタのグリッドセルごとの推計人口を表示する
- 独自のカラーマップを適用する
- 選択した範囲を行政界とともに表示する
- 完成した地図をHTMLファイルとして保存する

Workflow

1. Define the input and output paths
2. Define and read a 3,000 × 3,000 pixel raster window
3. Preserve NoData areas as transparent pixels
4. Inspect the selected window and its geographic bounds
5. Create and normalise the custom colour map
6. Create a light basemap without built-in place labels
7. Add the raster window, administrative boundaries and labels
8. Add the colour legend and information panel
9. Save and display the interactive map

Data

Population raster data:

- `worldpop_syria_2026.tif`

Source: WorldPop, 2026 estimated population dataset

Administrative boundary data:

- `syr_admin0.geojson`
- `syr_admin1.geojson`

Source: HDX OCHA, Syria subnational administrative boundaries

Data Scope and Limitations

- Population values are estimates rather than census counts.
- The source raster resolution is 3 arc seconds, approximately 100 metres.
- Only a 3,000 × 3,000 pixel window from the upper-left section of the raster is displayed.
- The selected window primarily covers northwestern Syria and does not represent the complete national raster.
- No spatial aggregation or resampling is applied to the selected window.
- Values above 100 are displayed using the highest colour class.
- The 0–100 display range changes only the map colours and does not modify the source raster values.
- The display layer should not be used independently to calculate the total population of Syria.

- 人口値は国勢調査による実測値ではなく推計値です。
- 元ラスタの解像度は3秒角で、約100メートルです。
- 元ラスタ左上の3,000 × 3,000ピクセルのみを表示しています。
- 選択範囲は主にシリア北西部を対象としており、シリア全土を表すものではありません。
- 選択した範囲には、集約処理やリサンプリングを行っていません。
- 100を超える値は、最上位の色で表示します。
- 0〜100の表示範囲は地図上の色だけに適用され、元ラスタの値は変更しません。
- 表示レイヤーだけを用いて、シリア全体の人口総数を計算することはできません。

Technologies

- Python
- Rasterio
- NumPy
- Folium
- Matplotlib
- Branca
- pathlib

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

from pathlib import Path

import branca
import folium
import matplotlib.colors as mcolors
import numpy as np
import rasterio
from rasterio.windows import Window

In [ ]:
# 2
# Define file paths
# ファイルパスを設定する

PROJECT_DIR = Path.cwd()
ROOT_DIR = PROJECT_DIR.parents[1]

VECTOR_DIR = ROOT_DIR / "02_DATA" / "VECTOR"
RASTER_DIR = ROOT_DIR / "02_DATA" / "RASTER"

admin0_path = VECTOR_DIR / "syr_admin0.geojson"
admin1_path = VECTOR_DIR / "syr_admin1.geojson"

population_path = RASTER_DIR / "worldpop_syria_2026.tif"

output_path = PROJECT_DIR / "02_syria_windowed_reading.html"

In [ ]:
# 3
# Define and read the population raster window
# 人口ラスタの読込範囲を指定し、データを取得する

WINDOW_WIDTH = 3000
WINDOW_HEIGHT = 3000

with rasterio.open(
    population_path
) as src:

    if src.crs is None:
        raise ValueError(
            "The population raster has no defined CRS."
        )

    if src.crs.to_epsg() != 4326:
        raise ValueError(
            "The population raster must use EPSG:4326 "
            "for Folium display."
        )

    if (
        src.width < WINDOW_WIDTH
        or src.height < WINDOW_HEIGHT
    ):
        raise ValueError(
            "The requested 3,000 × 3,000 pixel window "
            "exceeds the source raster dimensions."
        )

    # Define the window from the upper-left raster corner.
    # ラスタ左上を起点として読込範囲を指定する

    my_window = Window(
        col_off=0,
        row_off=0,
        width=WINDOW_WIDTH,
        height=WINDOW_HEIGHT,
    )

    # Read band 1 while masking NoData pixels.
    # NoData画素をマスクしながらバンド1を読み込む

    data_window = src.read(
        1,
        window=my_window,
        masked=True,
    ).astype(
        "float32"
    )

    # Retain the source NoData value and window bounds.
    # 元ラスタのNoData値と読込範囲を取得する

    nodata = src.nodata

    lon_left, lat_bottom, lon_right, lat_top = (
        src.window_bounds(
            my_window
        )
    )

In [ ]:
# 4
# Prepare and inspect the selected raster window
# 選択したラスタ範囲を表示用に整え、内容を確認する

# Convert masked NoData pixels to NaN.
# マスクされたNoData画素をNaNへ変換する

data_window_clean = data_window.filled(
    float("nan")
)

valid_window_pixels = np.isfinite(
    data_window_clean
)

if not valid_window_pixels.any():
    raise ValueError(
        "The selected raster window contains "
        "no valid population values."
    )

print(
    f"NoData value: {nodata}"
)

print(
    f"Window size: {data_window_clean.shape}"
)

print(
    "Valid window pixels: "
    f"{np.count_nonzero(valid_window_pixels):,}"
)

print(
    "Minimum population value: "
    f"{np.nanmin(data_window_clean):,.2f}"
)

print(
    "Maximum population value: "
    f"{np.nanmax(data_window_clean):,.2f}"
)

print(
    "Window bounds: "
    f"{lon_left}, {lat_bottom}, "
    f"{lon_right}, {lat_top}"
)

In [ ]:
# 5
# Create and normalise the custom colour map
# カラーマップを作成し、人口値を表示範囲へ正規化する

# Define the custom green colour palette.
# QGISで作成したグリーンカラーパレットを定義する

colors = [
    (0.00, "#7bb5a000"),
    (0.20, "#7bb5a0ff"),
    (0.40, "#2e9166ff"),
    (0.60, "#226c4cff"),
    (0.80, "#184d36ff"),
    (1.00, "#0f3122ff")
]

marisa_cmap = mcolors.LinearSegmentedColormap.from_list(
    "Syria_Windowed_Green",
    colors
)

# Make missing raster values transparent.
# 欠損値を透明にする

marisa_cmap.set_bad(
    (0, 0, 0, 0)
)

# Display population values from 0 to 100.
# 100を超える値は最高色として表示する

POPULATION_MIN = 0.0
POPULATION_MAX = 100.0

population_norm = mcolors.Normalize(
    vmin=POPULATION_MIN,
    vmax=POPULATION_MAX,
    clip=True
)


def population_colormap(value):
    """
    Convert a WorldPop value to an RGBA colour.
    WorldPopの人口値をRGBAカラーへ変換する。
    """

    # NaN represents a NoData pixel.
    # NaNはNoData画素を表すため、透明にする

    if value is None or value != value:
        return (0, 0, 0, 0)

    return marisa_cmap(
        population_norm(value)
    )

In [ ]:
# 6
# Create the focused base map
# シリア北西部を中心としたベースマップを作成する

m = folium.Map(
    location=[35.75, 36.90],
    zoom_start=8,
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "light_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap</a> contributors "
        '&copy; <a href="https://carto.com/attributions">CARTO</a>'
    )
)

# Define the initial viewport for northwestern Syria.
# Aleppo、Idleb、Lattakia、Hama、Tartousを含む初期表示範囲を設定する

focus_bounds = [
    [34.55, 35.55],
    [37.00, 38.40]
]

m.fit_bounds(
    focus_bounds,
    padding=(40, 40),
    max_zoom=9
)

In [ ]:
# 7
# Add the selected population raster window
# 選択した人口ラスタの範囲を地図へ重ねる

bounds_window = [
    [lat_bottom, lon_left],
    [lat_top, lon_right]
]

folium.raster_layers.ImageOverlay(
    image=data_window_clean,
    bounds=bounds_window,
    colormap=population_colormap,
    opacity=0.8,
    name="Population Window (3,000 × 3,000 pixels)"
).add_to(m)

In [ ]:
# 8
# Add administrative boundaries
# 国境線と県境線を追加する

folium.GeoJson(
    str(admin0_path),
    name="Country Boundary",
    style_function=lambda feature: {
        "color": "black",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m)

folium.GeoJson(
    str(admin1_path),
    name="Governorate Boundaries",
    style_function=lambda feature: {
        "color": "gray",
        "weight": 1,
        "fillOpacity": 0
    }
).add_to(m)

In [ ]:
# 9
# Add neighbouring country labels
# 隣国名を追加する

neighbors = {
    "TÜRKIYE": [37.5, 37.5],
    "IRAQ": [34.5, 42.0],
    "JORDAN": [31.9, 36.5],
    "LEBANON": [34.2, 35.0]
}

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 14pt;
                font-weight: bold;
                color: gray;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
            ">
                {name}
            </div>
            """
        )
    ).add_to(m)

In [ ]:
# 10
# Add governorate labels
# シリア14県の名称を追加する

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 10pt;
                color: black;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
            ">
                {name}
            </div>
            """
        )
    ).add_to(m)

In [ ]:
# 11
# Add population colour legend
# 人口値を示すカラーバーを追加する

legend_values = [
    0,
    20,
    40,
    60,
    80,
    100
]

legend_colors = [
    marisa_cmap(
        population_norm(value)
    )
    for value in legend_values
]

colormap = branca.colormap.LinearColormap(
    colors=legend_colors,
    index=legend_values,
    vmin=POPULATION_MIN,
    vmax=POPULATION_MAX,
    caption=(
        "Estimated population per source "
        "100 m grid cell"
    )
)

m.add_child(colormap)


# Style the legend for the light basemap.
# 白色ベースマップに合わせて凡例を整える

light_legend_css = """
<style>
    .legend {
        background: rgba(255, 255, 255, 0.9) !important;
        color: black !important;
        padding: 12px !important;
        border: 1px solid rgba(0, 0, 0, 0.4) !important;
        border-radius: 8px !important;
    }

    .legend text {
        fill: black !important;
        font-size: 12px !important;
        font-weight: bold !important;
    }

    .legend caption {
        color: black !important;
        font-weight: bold !important;
    }
</style>
"""

m.get_root().header.add_child(
    folium.Element(light_legend_css)
)

In [ ]:
# 12
# Add map information panel
# 地図の説明と出典を表示する情報パネルを追加する

title_html = """
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 390px;
    min-height: 175px;
    background-color: rgba(255, 255, 255, 0.92);
    color: black;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid #555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="font-size: 16px;">
        Syria
    </b>
    <br>

    <span style="
        color: #226c4c;
        font-weight: bold;
    ">
        Population Windowed Reading (2026)
    </span>
    <br>

    <small style="
        display: block;
        margin-top: 6px;
        line-height: 1.25;
        color: #333;
    ">
        A 3,000 × 3,000 pixel window is read from the upper-left section of the WorldPop raster without loading the complete dataset. The selected window primarily covers northwestern Syria. Values represent estimated population per source grid cell at approximately 100 m resolution.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 6px;
        border-top: 1px solid #aaa;
        font-size: 11px;
        color: #555;
    ">
        Source:
        <a
            href="https://hub.worldpop.org/geodata/summary?id=75632"
            target="_blank"
            style="
                color: #226c4c;
                text-decoration: none;
                font-weight: bold;
            "
        >
            WorldPop (Open Access Data)
        </a>
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(title_html)
)

In [ ]:
# 13
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(
    output_path
)

print(
    f"Map saved to: {output_path}"
)

m